In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from scipy.spatial.distance import cosine, pdist

In [19]:
CT_COL     = "TCRClonotype"
SAMPLE_COL = "Sample_Origin"
LY49C_COL  = "Ly-49C-Ly-49I-Klra3-Klra9-AMM2139-pAbO"

min_size = 5
plots_per_page = 16
thresholds = np.linspace(1.0, 0.85, 16)

In [20]:
markers_b10br = [
    "RiO-Allo:H-2Kb-ATLVFHNL-pAbO",
    "RiO-Allo:H-2Kb-EEEPVKKI-pAbO",
    "RiO-Allo:H-2Kb-HIYEFPQL-pAbO",
    "RiO-Allo:H-2Kb-INFDFPKL-pAbO",
    "RiO-Allo:H-2Kb-RAYLFNSV-pAbO",
    "RiO-Allo:H-2Kb-RTYTYEKL-pAbO",
    "RiO-Allo:H-2Kb-SNYLFTKL-pAbO",
    "RiO-Allo:H-2Kb-SSYTFPKM-pAbO",
    "RiO-Allo:H-2Kb-SVYVYKVL-pAbO",
    "RiO-Allo:H-2Kb-VAFDFTKV-pAbO",
    "RiO-Allo:H-2Kb-VGPRYTNL-pAbO",
    "RiO-Allo:H-2Kb-VIVRFLTV-pAbO",
    "RiO-Allo:H-2Kb-VSFTYRYL-pAbO",
]

peptides_b10br = [
    "ATLVFHNL","EEEPVKKI","HIYEFPQL","INFDFPKL","RAYLFNSV","RTYTYEKL",
    "SNYLFTKL","SSYTFPKM","SVYVYKVL","VAFDFTKV","VGPRYTNL","VIVRFLTV","VSFTYRYL"
]

markers_balbc = [
    "RiO-Allo:H-2Kb-ATLVFHNL-pAbO",
    "RiO-Allo:H-2Kb-HIYEFPQL-pAbO",
    "RiO-Allo:H-2Kb-INFDFPKL-pAbO",
    "RiO-Allo:H-2Kb-RAYLFNSV-pAbO",
    "RiO-Allo:H-2Kb-RTYTYEKL-pAbO",
    "RiO-Allo:H-2Kb-SNYLFTKL-pAbO",
    "RiO-Allo:H-2Kb-SSYTFPKM-pAbO",
    "RiO-Allo:H-2Kb-SVYVYKVL-pAbO",
    "RiO-Allo:H-2Kb-VAFDFTKV-pAbO",
    "RiO-Allo:H-2Kb-VGPRYTNL-pAbO",
    "RiO-Allo:H-2Kb-VIVRFLTV-pAbO",
    "RiO-Allo:H-2Kb-VSFTYRYL-pAbO",
    "RiO-H-2:H-2Kd-SYFPEITHI-ADEX5099-pAbO"
]

peptides_balbc = [
    "ATLVFHNL","HIYEFPQL","INFDFPKL","RAYLFNSV","RTYTYEKL","SNYLFTKL",
    "SSYTFPKM","SVYVYKVL","VAFDFTKV","VGPRYTNL","VIVRFLTV","VSFTYRYL","SYFPEITHI"
]

In [21]:
def row_normalise(X: np.ndarray) -> np.ndarray:
    X = np.asarray(X, dtype=float)
    rs = X.sum(axis=1, keepdims=True)
    rs[rs == 0] = 1.0
    return X / rs

In [22]:
def renyi_entropy(P: np.ndarray, alpha: float, axis: int = -1, eps: float = 1e-300) -> np.ndarray:
    """
    Renyi entropy H_alpha for probability vectors P along `axis`.

    - alpha = 1 gives Shannon (limit).
    - alpha = 0 gives log(k) where k = number of nonzero entries (Hartley).
    - alpha -> inf would be -log(max p), but we only need 0..5.

    Returns entropy in bits (log2).
    """
    P = np.asarray(P, dtype=float)

    if alpha == 1.0:
        # Shannon (Renyi order 1)
        with np.errstate(divide="ignore", invalid="ignore"):
            logP = np.where(P > 0, np.log2(P), 0.0)
        return -(P * logP).sum(axis=axis)

    if alpha == 0.0:
        # Renyi order 0
        k = np.sum(P > 0, axis=axis)
        k = np.maximum(k, 1)
        return np.log2(k)

    # Renyi order alpha
    S = np.sum(np.power(np.maximum(P, 0.0), alpha), axis=axis)
    S = np.maximum(S, eps)
    return (1.0 / (1.0 - alpha)) * np.log2(S)

In [23]:
# Coherence (dot product)
def mean_pairwise_cosine_similarity(P: np.ndarray) -> float:
    """
    P is already row-normalised per cell (n_cells x n_peptides).
    Returns mean pairwise cosine similarity, or NaN if <2 cells.
    """
    if P.shape[0] < 2:
        return np.nan
    return float(np.mean(1.0 - pdist(P, metric="cosine")))

In [24]:
def compute_clonotype_summaries(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
    entropy_alpha: float = 1.0,
) -> dict:
    """
    Returns dict ct -> summary with:
      - mean_pattern: mean of per-cell normalised RiO vectors
      - mean_entropy: mean Renyi entropy (alpha) across cells
      - coherence: mean pairwise cosine similarity across cells (on per-cell normalised)
      - n_cells: number of cells
    """
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]
    out = {}

    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        P = row_normalise(X)

        ent = float(np.mean(renyi_entropy(P, alpha=entropy_alpha, axis=1)))
        coh = mean_pairwise_cosine_similarity(P)
        out[ct] = {
            "mean_pattern": P.mean(axis=0),
            "mean_entropy": ent,
            "coherence": coh,
            "n_cells": int(P.shape[0]),
        }
    return out

In [25]:
# Filtering within a sample
def apply_ly49c_filter_one_sample(df_s: pd.DataFrame, ly49c_col: str, q: float) -> tuple[pd.DataFrame, float]:
    """
    Within ONE sample: keep Ly49C <= quantile_q.
    Returns (filtered_df, Ts).
    """
    if q >= 1.0:
        return df_s.copy(), float(df_s[ly49c_col].max())
    Ts = float(df_s[ly49c_col].quantile(q))
    return df_s[df_s[ly49c_col] <= Ts].copy(), Ts

In [26]:
# Baseline vs filtered
def pattern_distance(p1: np.ndarray, p2: np.ndarray, metric: str) -> float:
    if metric == "cosine":
        if np.allclose(p1, 0) or np.allclose(p2, 0):
            return 1.0
        return float(cosine(p1, p2))
    if metric == "l1":
        return float(np.sum(np.abs(p1 - p2)))
    raise ValueError(f"Unknown metric: {metric}")

In [27]:
# Sweeping within a sample
def ly49c_sweep_metrics_one_sample(
    df_s: pd.DataFrame,
    ct_col: str,
    ly49c_col: str,
    markers: list[str],
    thresholds: np.ndarray,
    min_size: int = 5,
    entropy_alpha: float = 1.0,
) -> pd.DataFrame:
    """
    Computes sweep metrics for ONE sample dataset.

    Metrics include:
      - retention
      - Δ entropy (Renyi alpha; alpha=1 => Shannon)
      - Δ coherence
      - Δ cosine distance (median over common clonotypes)
      - Δ L1 distance (median over common clonotypes)
    """
    baseline = compute_clonotype_summaries(df_s, ct_col, markers, min_size=min_size, entropy_alpha=entropy_alpha)
    n0 = len(df_s)

    rows = []
    for q in thresholds:
        df_f, Ts = apply_ly49c_filter_one_sample(df_s, ly49c_col, float(q))
        filtered = compute_clonotype_summaries(df_f, ct_col, markers, min_size=min_size, entropy_alpha=entropy_alpha)

        common = set(baseline) & set(filtered)
        if common:
            cos_d = [pattern_distance(baseline[c]["mean_pattern"], filtered[c]["mean_pattern"], "cosine") for c in common]
            l1_d  = [pattern_distance(baseline[c]["mean_pattern"], filtered[c]["mean_pattern"], "l1") for c in common]
            ent_d = [filtered[c]["mean_entropy"] - baseline[c]["mean_entropy"] for c in common]
            coh_d = [filtered[c]["coherence"] - baseline[c]["coherence"] for c in common]

            row = {
                "percentile": float(q),
                "Ts_ly49c": Ts,
                "n_cells": int(len(df_f)),
                "pct_cells_retained": 100.0 * len(df_f) / n0 if n0 else np.nan,
                "n_clonotypes_baseline_ge5": int(len(baseline)),
                "n_clonotypes_filtered_ge5": int(len(filtered)),
                "n_common_clonotypes_ge5": int(len(common)),
                "median_cosine_dist": float(np.median(cos_d)),
                "median_l1_dist": float(np.median(l1_d)),
                "mean_entropy_change": float(np.mean(ent_d)),
                "mean_coherence_change": float(np.nanmean(coh_d)),
            }
        else:
            row = {
                "percentile": float(q),
                "Ts_ly49c": Ts,
                "n_cells": int(len(df_f)),
                "pct_cells_retained": 100.0 * len(df_f) / n0 if n0 else np.nan,
                "n_clonotypes_baseline_ge5": int(len(baseline)),
                "n_clonotypes_filtered_ge5": int(len(filtered)),
                "n_common_clonotypes_ge5": 0,
                "median_cosine_dist": np.nan,
                "median_l1_dist": np.nan,
                "mean_entropy_change": np.nan,
                "mean_coherence_change": np.nan,
            }

        rows.append(row)

    return pd.DataFrame(rows)

In [28]:
# Baseline and filtered (raw counts and proportions)
def clonotype_raw_and_prop_from_raw(
    df: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    min_size: int = 5,
) -> tuple[dict, dict, dict]:
    """
    For clonotypes with >=min_size cells:
      raw_sum[ct] = sum of raw peptide counts across cells
      prop[ct]    = raw_sum / raw_sum.sum()
      n_cells[ct] = number of cells in clonotype (baseline)
    """
    vc = df[ct_col].value_counts()
    keep = vc[vc >= min_size].index
    df2 = df[df[ct_col].isin(keep)]

    raw_sum, prop, n_cells = {}, {}, {k: int(v) for k, v in vc[keep].to_dict().items()}
    for ct, g in df2.groupby(ct_col):
        X = g[markers].to_numpy(dtype=float, copy=False)
        s = X.sum(axis=0)
        raw_sum[ct] = s
        tot = s.sum()
        prop[ct] = s / tot if tot > 0 else np.zeros_like(s)

    return raw_sum, prop, n_cells

In [29]:
def plot_raw_and_prop_before_after_4x8_pdf(
    raw_b: dict, raw_a: dict,
    prop_b: dict, prop_a: dict,
    n_b: dict, n_a: dict,
    peptides: list[str],
    out_pdf: Path,
    title: str,
):
    """
    4x8 panels per page (32 axes).
    Each clonotype uses 2 axes: RAW (left) + PROP (right).
    => 16 clonotypes per page.
    """
    xs = np.arange(len(peptides))
    w = 0.42
    per_page = 16

    clonotypes = sorted(n_b.keys(), key=lambda c: n_b[c], reverse=True)

    with PdfPages(out_pdf) as pdf:
        for start in range(0, len(clonotypes), per_page):
            chunk = clonotypes[start:start + per_page]

            fig, axes = plt.subplots(4, 8, figsize=(24, 12))
            axes = axes.flatten()

            for i, ct in enumerate(chunk):
                ax_raw = axes[2*i]
                ax_prp = axes[2*i + 1]
                has_after = ct in raw_a

                # Raw
                b = raw_b[ct]
                if has_after:
                    a = raw_a[ct]
                    ax_raw.bar(xs - w/2, b, w, alpha=0.8, label="Before")
                    ax_raw.bar(xs + w/2, a, w, alpha=0.8, label="After")
                    ax_raw.set_title(f"{ct}\nRAW n:{n_b[ct]}→{n_a.get(ct,0)}", fontsize=7)
                else:
                    ax_raw.bar(xs, b, w*1.8, alpha=0.6, label="Before (lost)")
                    ax_raw.set_title(f"{ct}\nRAW n:{n_b[ct]}→0 LOST", fontsize=7, color="darkred")

                ax_raw.set_xticks(xs)
                ax_raw.set_xticklabels(peptides, rotation=90, fontsize=6)
                ax_raw.tick_params(axis="y", labelsize=6)
                ax_raw.spines["top"].set_visible(False)
                ax_raw.spines["right"].set_visible(False)
                if i == 0:
                    ax_raw.legend(fontsize=7, loc="upper right")

                # Proportions
                b = prop_b[ct]
                if has_after:
                    a = prop_a[ct]
                    ax_prp.bar(xs - w/2, b, w, alpha=0.8, label="Before")
                    ax_prp.bar(xs + w/2, a, w, alpha=0.8, label="After")
                    ax_prp.set_title(f"{ct}\nPROP n:{n_b[ct]}→{n_a.get(ct,0)}", fontsize=7)
                else:
                    ax_prp.bar(xs, b, w*1.8, alpha=0.6, label="Before (lost)")
                    ax_prp.set_title(f"{ct}\nPROP n:{n_b[ct]}→0 LOST", fontsize=7, color="darkred")

                ax_prp.set_xticks(xs)
                ax_prp.set_xticklabels(peptides, rotation=90, fontsize=6)
                ax_prp.tick_params(axis="y", labelsize=6)
                ax_prp.spines["top"].set_visible(False)
                ax_prp.spines["right"].set_visible(False)

            for j in range(2 * len(chunk), 32):
                axes[j].axis("off")

            page = start // per_page + 1
            n_pages = (len(clonotypes) + per_page - 1) // per_page
            fig.suptitle(f"{title} (page {page}/{n_pages})", fontsize=14, y=0.995)
            fig.tight_layout(rect=[0, 0, 1, 0.97])
            pdf.savefig(fig)
            plt.close(fig)

In [30]:
# Renyi entropy orders 0->5 for top 10 clonotypes per sample
def renyi_profile_top_clonotypes(
    df_s: pd.DataFrame,
    ct_col: str,
    markers: list[str],
    top_n: int = 10,
    orders: list[float] | None = None,
) -> pd.DataFrame:
    """
    For the top-N most abundant clonotypes in ONE sample:
      compute mean Renyi entropy per clonotype for each alpha in `orders`.
    Returns long-form table: clonotype, n_cells, alpha, renyi_entropy_mean
    """
    if orders is None:
        orders = [0, 0.5, 1, 2, 3, 4, 5]

    vc = df_s[ct_col].value_counts()
    top_cts = list(vc.index[:top_n])

    rows = []
    for ct in top_cts:
        g = df_s[df_s[ct_col] == ct]
        X = g[markers].to_numpy(dtype=float, copy=False)
        P = row_normalise(X)
        n_cells = int(P.shape[0])

        for a in orders:
            h = float(np.mean(renyi_entropy(P, alpha=float(a), axis=1)))
            rows.append({"clonotype": ct, "n_cells": n_cells, "alpha": float(a), "renyi_entropy_mean": h})

    return pd.DataFrame(rows)

In [31]:
def plot_renyi_profiles(df_long: pd.DataFrame, out_png: Path, title: str):
    """
    Simple line plot per clonotype: alpha vs entropy.
    """
    fig, ax = plt.subplots(figsize=(8, 6))
    for ct, g in df_long.groupby("clonotype"):
        g2 = g.sort_values("alpha")
        ax.plot(g2["alpha"], g2["renyi_entropy_mean"], marker="o", label=str(ct))
    ax.set_xlabel("Renyi order α")
    ax.set_ylabel("Mean Renyi entropy (bits)")
    ax.set_title(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [32]:
def plot_sweep_metrics_2x3(
    sweep_df: pd.DataFrame,
    out_png: Path,
    title: str,
    chosen_q: float
):
    fig, axes = plt.subplots(2, 3, figsize=(14, 9))
    axes = axes.flatten()
    x = sweep_df["percentile"]

    panels = [
        ("pct_cells_retained",      "% retained",        "Retention"),
        ("median_cosine_dist",      "median",            "Δ cosine distance"),
        ("median_l1_dist",          "median",            "Δ L1 distance"),
        ("mean_entropy_change",     "mean Δ",            "Δ entropy (Renyi/Shannon)"),
        ("mean_coherence_change",   "mean Δ",            "Δ coherence"),
        ("Ts_ly49c",                "Ts",                "Ly49C threshold Ts"),
    ]

    for ax, (col, ylab, ttl) in zip(axes, panels):
        ax.plot(x, sweep_df[col], marker="o", lw=2, ms=4)
        ax.axvline(chosen_q, color="red", ls="--", alpha=0.7)
        if "change" in col:
            ax.axhline(0, color="gray", ls="--", alpha=0.5)

        ax.set_xlabel("q")
        ax.set_ylabel(ylab)
        ax.set_title(ttl)
        ax.invert_xaxis()
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)

    fig.suptitle(title, fontsize=14, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.97])
    fig.savefig(out_png, dpi=170, bbox_inches="tight")
    plt.close(fig)

In [33]:
def run_samplewise_pipeline(
    df: pd.DataFrame,
    dataset_name: str,
    markers: list[str],
    peptides: list[str],
    output_root: Path,
    thresholds: np.ndarray,
    chosen_q: float,
    ct_col: str = CT_COL,
    sample_col: str = SAMPLE_COL,
    ly49c_col: str = LY49C_COL,
    min_size: int = 5,
    entropy_alpha_for_sweep: float = 1.0,
    renyi_orders: list[float] | None = None
) -> pd.DataFrame:
    """
    Creates output folders:
      output_root/dataset_name/<sample>/
        - ly49c_sweep_metrics_alpha{alpha}.csv
        - clonotypes_RAW_and_PROP_before_after_q{chosen_q}.pdf
        - renyi_top10.csv
        - renyi_top10.png (optional)
      output_root/dataset_name/INDEX.csv
    """
    output_root = Path(output_root)
    ds_out = output_root / dataset_name
    ds_out.mkdir(parents=True, exist_ok=True)

    needed = {ct_col, sample_col, ly49c_col, *markers}
    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise KeyError(f"Missing required columns (first few): {missing[:10]}")

    index_rows = []

    for sample, df_s in df.groupby(sample_col):
        s_out = ds_out / str(sample)
        s_out.mkdir(parents=True, exist_ok=True)

        # Sweeps
        sweep_df = ly49c_sweep_metrics_one_sample(
            df_s=df_s,
            ct_col=ct_col,
            ly49c_col=ly49c_col,
            markers=markers,
            thresholds=thresholds,
            min_size=min_size,
            entropy_alpha=entropy_alpha_for_sweep,
        )
        sweep_csv = s_out / f"ly49c_sweep_metrics_alpha{entropy_alpha_for_sweep}.csv"
        sweep_df.to_csv(sweep_csv, index=False)

        sweep_plot = s_out / f"ly49c_sweep_metrics_alpha{entropy_alpha_for_sweep}.png"
        plot_sweep_metrics_2x3(
            sweep_df=sweep_df,
            out_png=sweep_plot,
            title=f"{dataset_name} | {sample} | sweep (alpha={entropy_alpha_for_sweep})",
            chosen_q=chosen_q
        )


        # Baseline and filtered (raw counts and proportional)
        df_after, Ts = apply_ly49c_filter_one_sample(df_s, ly49c_col, chosen_q)
        raw_b, prop_b, n_b = clonotype_raw_and_prop_from_raw(df_s, ct_col, markers, min_size=min_size)
        raw_a, prop_a, n_a = clonotype_raw_and_prop_from_raw(df_after, ct_col, markers, min_size=min_size)

        pdf_out = s_out / f"clonotypes_RAW_and_PROP_before_after_q{chosen_q:.3f}.pdf"
        plot_raw_and_prop_before_after_4x8_pdf(
            raw_b, raw_a, prop_b, prop_a, n_b, n_a,
            peptides=peptides,
            out_pdf=pdf_out,
            title=f"{dataset_name} | {sample} | clonotypes≥{min_size} RAW+PROP before/after (q={chosen_q:.3f}, Ts={Ts:.2f})",
        )

        # Renyis order 0->5
        renyi_df_base = renyi_profile_top_clonotypes(
            df_s=df_s, ct_col=ct_col, markers=markers, top_n=10, orders=renyi_orders or [0,1,2,3,4,5]
        )
        renyi_df_base["condition"] = "baseline"

        renyi_df_after = renyi_profile_top_clonotypes(
            df_s=df_after, ct_col=ct_col, markers=markers, top_n=10, orders=renyi_orders or [0,1,2,3,4,5]
        )
        renyi_df_after["condition"] = f"filtered_q{chosen_q:.3f}"

        renyi_df = pd.concat([renyi_df_base, renyi_df_after], ignore_index=True)
        renyi_csv = s_out / f"renyi_top10_alpha0to5_baseline_and_filtered_q{chosen_q:.3f}.csv"
        renyi_df.to_csv(renyi_csv, index=False)

        for cond, sub in renyi_df.groupby("condition"):
            out_png = s_out / f"renyi_top10_{cond}.png"
            plot_renyi_profiles(sub, out_png, f"{dataset_name} | {sample} | Renyi top10 ({cond})")

        index_rows.append({
            "dataset": dataset_name,
            "sample": sample,
            "n_cells_before": len(df_s),
            "n_cells_after": len(df_after),
            "n_clonotypes_ge5_before": len(n_b),
            "n_clonotypes_ge5_after": len(n_a),
            "chosen_q": chosen_q,
            "Ts": Ts,
            "sweep_csv": str(sweep_csv),
            "sweep_plot": str(sweep_plot),
            "before_after_pdf": str(pdf_out),
            "renyi_csv": str(renyi_csv),
        })

    index_df = pd.DataFrame(index_rows)
    index_df.to_csv(ds_out / "INDEX.csv", index=False)
    return index_df

In [34]:
df_b10br = pd.read_csv(
    "../Data/20260116 Comparison 3/20251223 BL6-B10BR HTx HIL Clonotypes with ADT Counts.csv",
    index_col=0
)

df_balbc = pd.read_csv(
    "../Data/20260116 Comparison 3/20251218 BL6-BALBc HTx HIL Clonotypes with ADT Counts.csv",
    index_col=0
)

In [ ]:
out_b10br = run_samplewise_pipeline(
    df=df_b10br,
    dataset_name="B10BR_HIL",
    markers=markers_b10br,
    peptides=peptides_b10br,
    output_root=Path("Comparison3_Samplewise_Outputs"),
    thresholds=thresholds,
    chosen_q=0.925,
    min_size=5,
    entropy_alpha_for_sweep=1.0,   # Shannon for sweep
    renyi_orders=[0, 1, 2, 3, 4, 5]
)

out_balbc = run_samplewise_pipeline(
    df=df_balbc,
    dataset_name="BALBc_HIL",
    markers=markers_balbc,
    peptides=peptides_balbc,
    output_root=Path("Comparison3_Samplewise_Outputs"),
    thresholds=thresholds,
    chosen_q=0.925,
    min_size=5,
    entropy_alpha_for_sweep=1.0,
    renyi_orders=[0, 1, 2, 3, 4, 5]
)